# Brain Tumor Detection — Step 2: Transfer Learning

**Goal:** beat the Step 1 baseline CNN using a pretrained backbone (EfficientNetB0) plus data augmentation, trained in two phases:

1. **Feature extraction** — freeze the pretrained backbone, train only a new classification head
2. **Fine-tuning** — unfreeze the top layers of the backbone and keep training at a very low learning rate

This notebook is self-contained (re-downloads the dataset), so it can run independently of the Step 1 notebook. Bring your Step 1 test accuracy with you to compare at the end.

**Note on image size:** EfficientNetB0 expects `224x224` inputs (vs. `150x150` used in the baseline) — that's a requirement of the pretrained weights, not a free choice.


## 1. Setup

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Get the dataset

In [ ]:
!pip install -q kagglehub

import kagglehub

dataset_path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
print("Dataset downloaded to:", dataset_path)

train_dir = f"{dataset_path}/Training"
test_dir = f"{dataset_path}/Testing"


## 3. Load the data

Images are loaded at `224x224` and left as raw `[0, 255]` pixel values — EfficientNet's own `preprocess_input` (applied inside the model, not here) expects that range, not a manual `/255` rescale.


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", class_names)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)
test_ds = test_ds.cache().prefetch(AUTOTUNE)


## 4. Data augmentation

Applied as model layers, so it only runs during training (`training=True`) and is automatically skipped at inference/evaluation — no manual on/off switching needed later in the Streamlit app.


In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="augmentation")


## 5. Build the model

`EfficientNetB0` pretrained on ImageNet, backbone frozen for now. Preprocessing (`efficientnet.preprocess_input`) is baked into the model graph so the exact same model file can take raw images later in the Streamlit app — no separate preprocessing code to keep in sync.


In [ ]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMG_SIZE, 3),
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


## 6. Phase 1 — train the head (backbone frozen)


In [ ]:
callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
]

EPOCHS_PHASE1 = 12

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
)


## 7. Phase 2 — fine-tune

Unfreeze the top layers of the backbone and continue training at a much lower learning rate, so the pretrained weights adjust gently instead of being overwritten.


In [ ]:
base_model.trainable = True

# Freeze everything except the last ~30 layers of the backbone
FINE_TUNE_AT = len(base_model.layers) - 30
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()


In [ ]:
callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
]

EPOCHS_PHASE2 = 10
total_epochs = EPOCHS_PHASE1 + EPOCHS_PHASE2

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=total_epochs,
    initial_epoch=history1.epoch[-1] + 1,
    callbacks=callbacks_phase2,
)


### Training curves (both phases combined)

In [ ]:
acc = history1.history["accuracy"] + history2.history["accuracy"]
val_acc = history1.history["val_accuracy"] + history2.history["val_accuracy"]
loss = history1.history["loss"] + history2.history["loss"]
val_loss = history1.history["val_loss"] + history2.history["val_loss"]
epochs_range = range(len(acc))

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label="Train")
plt.plot(epochs_range, val_acc, label="Val")
plt.axvline(len(history1.history["accuracy"]) - 1, color="gray", linestyle="--", label="fine-tune starts")
plt.legend()
plt.title("Accuracy")

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label="Train")
plt.plot(epochs_range, val_loss, label="Val")
plt.axvline(len(history1.history["loss"]) - 1, color="gray", linestyle="--", label="fine-tune starts")
plt.legend()
plt.title("Loss")

plt.show()


## 8. Evaluate on the test set

Compare `test_acc` below against your Step 1 baseline number.


In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test accuracy: {test_acc:.4f}")
print(f"Test loss: {test_loss:.4f}")

y_true = np.concatenate([y for _, y in test_ds], axis=0)
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)

print(classification_report(y_true, y_pred, target_names=class_names))


### Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Transfer Learning (EfficientNetB0)")
plt.show()


## 9. Save the model

This is the model the Streamlit app will load, so save it somewhere that survives the runtime — Drive is recommended.


In [ ]:
model.save("brain_tumor_transfer.keras")
print("Saved brain_tumor_transfer.keras")

# Persist to Drive so it survives a runtime reset:
# from google.colab import drive
# drive.mount('/content/drive')
# model.save('/content/drive/MyDrive/brain_tumor_transfer.keras')


## If accuracy isn't beating the baseline

A few quick things to try, roughly in order of effort:
- Increase `FINE_TUNE_AT` window (unfreeze more layers) — currently unfreezes the last 30
- Try a different backbone: `tf.keras.applications.ResNet50` or `MobileNetV2` (swap the `preprocess_input` call to match)
- Increase augmentation strength if train accuracy is much higher than val accuracy (overfitting)
- Check class balance in the dataset — an uneven class distribution can inflate accuracy while hurting per-class recall (check the classification report above, not just the top-line accuracy)

## Next: Step 3

Wrap this saved model in a clean `predict(image) -> (label, confidence)` function, then move to the Streamlit interface (Step 4) and the medical info assistant (Step 5).
